# Probing Neural Networks — a hands-on tutorial

*Companion to the OXML Summer School 2026 lecture "Probing Neural Networks" (slides in this repo:
`OXML_Summer_School_2026_Lecture.pdf`).*

**The question of the whole field, in one line:** a language model gets things right on average — but
*what has it actually encoded inside*, and can we read it out?

In this notebook you will build the entire probing toolkit **on a real transformer (GPT-2)**, from
scratch, and — more importantly — learn to *distrust your own probe*. The skills that separate a
careful probing study from a misleading one are baselines, **selectivity**, and the gap between
*accessible* and *used*. You will implement all of them.

### How to use this notebook
- Cells marked **`# ✏️ EXERCISE`** have a piece missing (`raise NotImplementedError`). Fill it in.
- Right after each exercise is a **self-check** cell that `assert`s your answer is sensible. If it runs
  without error, you're good.
- Stuck? The solutions notebook (`probing_tutorial_solutions.ipynb`) has every answer filled in — but
  try first; the exercises are short and the point is the *thinking*.

### What you'll be able to do by the end
1. Extract frozen hidden states from a transformer and train a **linear probe**.
2. Read a **layer profile** (where in the network a concept lives).
3. Compute the **three baselines** and **selectivity** — and see why raw accuracy lies.
4. Show that a stronger (MLP) probe can be *more accurate but less trustworthy*.
5. Demonstrate the difference between **information being accessible** and the model **using** it
   (ablation and steering).
6. Point the same pipeline at **truthfulness/deception** — the basis of real safety probes.

> Runs free on Colab (`Runtime → Change runtime type → T4 GPU`), or on a laptop CPU in a few minutes.


## 0 · Setup

We use plain 🤗 `transformers` (no special interpretability library), `scikit-learn` for the probes,
and GPT-2 small — 12 layers, hidden size 768. Everything here transfers unchanged to bigger models.

In [ ]:
# If on Colab / fresh env, uncomment:
# !pip -q install transformers torch scikit-learn datasets matplotlib

import numpy as np, torch, matplotlib.pyplot as plt
from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer, AutoConfig

SEED = 0
np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "gpt2"                 # 124M params, d=768, 12 layers
print("device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModel.from_pretrained(MODEL_NAME).to(device).eval()
N_LAYERS = model.config.n_layer     # 12
D_MODEL  = model.config.n_embd      # 768
print(f"{MODEL_NAME}: {N_LAYERS} layers, hidden size {D_MODEL}")

# The single most important line in probing:
for p in model.parameters():
    p.requires_grad_(False)
# The encoder is FROZEN. It never updates. Only the probe will learn.
# Without this you are fine-tuning, not probing.

## 1 · A concept with clear labels: sentiment

We learn the *method* on a concept where the ground truth is unambiguous — the sentiment of a short
review (positive / negative). Later (§7) we point the exact same code at something you'd actually
deploy: whether a model is being **truthful**.

We load Stanford Sentiment Treebank (SST-2). If you're offline, a small built-in set is used instead
so the notebook still runs.

In [ ]:
def load_sentiment(n_train=400, n_test=200):
    try:
        from datasets import load_dataset
        ds = load_dataset("glue", "sst2")
        tr = ds["train"].shuffle(seed=SEED).select(range(n_train))
        te = ds["validation"].shuffle(seed=SEED).select(range(min(n_test, len(ds["validation"]))))
        return (list(tr["sentence"]), list(tr["label"]),
                list(te["sentence"]),  list(te["label"]))
    except Exception as e:
        print("Falling back to the built-in mini set (", type(e).__name__, ")")
        pos = ["a gorgeous, funny, deeply human film", "one of the best movies of the year",
               "I loved every minute of it", "sharp, tender and beautifully acted",
               "a triumph — moving and unforgettable", "witty, warm and wonderfully paced",
               "an absolute delight from start to finish", "smart, stylish and genuinely thrilling"]
        neg = ["a boring, lifeless mess", "two hours I will never get back",
               "poorly written and badly acted", "dull, clumsy and utterly forgettable",
               "a joyless slog with no payoff", "clichéd, tedious and painfully slow",
               "an incoherent waste of a good cast", "flat dialogue and zero tension"]
        X = pos*8 + neg*8; y = [1]*len(pos)*8 + [0]*len(neg)*8
        idx = np.random.permutation(len(X)); X = [X[i] for i in idx]; y = [y[i] for i in idx]
        cut = int(0.7*len(X))
        return X[:cut], y[:cut], X[cut:], y[cut:]

X_train, y_train, X_test, y_test = load_sentiment()
y_train, y_test = np.array(y_train), np.array(y_test)
print(f"{len(X_train)} train / {len(X_test)} test examples")
print("example:", repr(X_train[0]), "label:", y_train[0], "(1=positive, 0=negative)")

## 2 · Representations — turning text into $h \in \mathbb{R}^d$

For each input, the frozen model produces a hidden state at every layer. We pass
`output_hidden_states=True` and get back a tuple of `N_LAYERS + 1` tensors (embeddings + one per
block), each `[batch, seq_len, d]`.

We need **one vector per sentence**, so we **mean-pool over the real (non-padding) tokens**.

### ✏️ Exercise 1 — mean-pool the hidden states
Complete `get_activations`. Given the `hidden_states` at a chosen layer (shape `[B, T, d]`) and the
attention `mask` (shape `[B, T]`, 1 for real tokens), return the masked mean over tokens: shape
`[B, d]`. *Hint: multiply by the mask, sum over the token axis, divide by the number of real tokens.*

In [ ]:
@torch.no_grad()
def _forward(texts, batch_size=32):
    # Run the model; return (list of per-layer hidden states, attention masks) for a batch of texts.
    all_hs = [[] for _ in range(N_LAYERS + 1)]
    all_mask = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=64).to(device)
        out = model(**enc, output_hidden_states=True)
        for L, hs in enumerate(out.hidden_states):
            all_hs[L].append(hs.cpu())
        all_mask.append(enc["attention_mask"].cpu())
    # pad-concat is avoided by keeping per-batch; the pooling fn below handles each batch
    return all_hs, all_mask

In [ ]:
def masked_mean(hidden_states, mask):
    # hidden_states: [B, T, d]   mask: [B, T]  (1 = real token, 0 = padding)
    m = mask.unsqueeze(-1).to(hidden_states.dtype)      # [B, T, 1]
    summed = (hidden_states * m).sum(dim=1)             # [B, d]
    counts = m.sum(dim=1).clamp(min=1)                  # [B, 1]
    return summed / counts                              # [B, d]

def get_activations(texts, layer, batch_size=32):
    # Frozen hidden states at `layer`, mean-pooled to one [d] vector per text -> [n, d] numpy array.
    all_hs, all_mask = _forward(texts, batch_size)
    pooled = [masked_mean(hs, m) for hs, m in zip(all_hs[layer], all_mask)]
    return torch.cat(pooled, dim=0).float().numpy()


In [ ]:
# self-check
_H = get_activations(X_train[:5], layer=6)
assert _H.shape == (5, D_MODEL), f"expected (5, {D_MODEL}), got {_H.shape}"
assert np.isfinite(_H).all(), "activations contain NaN/inf"
print("✓ Exercise 1: get_activations returns", _H.shape)

## 3 · Your first linear probe

A **linear probe** is just logistic regression on the frozen activations:
$\hat y = \operatorname{softmax}(W h + b)$. If it succeeds, we infer the concept is *linearly
decodable* — encoded as a **direction** in $\mathbb{R}^d$.

We standardise features first (activations have very different scales across dimensions), which just
helps the optimiser — the probe is still linear in $h$.

### ✏️ Exercise 2 — train and evaluate a linear probe
Fill in `train_linear_probe` (fit `LogisticRegression` inside a `StandardScaler` pipeline) and
`accuracy` (fraction correct on held-out data).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

def train_linear_probe(X, y, C=1.0):
    probe = make_pipeline(StandardScaler(),
                          LogisticRegression(C=C, max_iter=2000))
    probe.fit(X, y)
    return probe

def accuracy(probe, X, y):
    return float((probe.predict(X) == y).mean())


In [ ]:
# extract once at a mid layer, train, evaluate
LAYER = 6
Xtr = get_activations(X_train, LAYER)
Xte = get_activations(X_test,  LAYER)
probe = train_linear_probe(Xtr, y_train)
acc = accuracy(probe, Xte, y_test)
print(f"layer {LAYER}: test accuracy = {acc:.3f}")
assert acc > 0.6, "a sentiment probe on a mid layer should beat 0.6 — check Exercise 2"
print("✓ Exercise 2")

## 4 · Probe *every* layer — the layer profile is a finding

Never report only your best layer (that's p-hacking). The *shape* of accuracy-vs-layer tells you
where the concept becomes linearly available.

### ✏️ Exercise 3 — sweep the layers
Fill in the loop: for each layer `0..N_LAYERS`, extract train/test activations, train a probe, record
test accuracy. (This recomputes activations per layer for clarity; in a real study you'd cache them.)

In [ ]:
def layer_profile(X_train, y_train, X_test, y_test):
    accs = []
    for L in range(N_LAYERS + 1):
        Xtr_L = get_activations(X_train, L)
        Xte_L = get_activations(X_test,  L)
        p = train_linear_probe(Xtr_L, y_train)
        accs.append(accuracy(p, Xte_L, y_test))
    return accs

profile = layer_profile(X_train, y_train, X_test, y_test)


In [ ]:
assert len(profile) == N_LAYERS + 1
plt.figure(figsize=(6,3.2))
plt.plot(range(N_LAYERS+1), profile, "o-", color="#13294B")
plt.xlabel("layer"); plt.ylabel("probe accuracy"); plt.title("Sentiment — layer profile")
plt.axhline(max(np.mean(y_test), 1-np.mean(y_test)), ls="--", c="grey", label="majority baseline")
plt.legend(); plt.tight_layout(); plt.show()
best = int(np.argmax(profile))
print(f"peak at layer {best} (acc {profile[best]:.3f})")
print("✓ Exercise 3 — note WHERE it peaks and whether it dips in the final layers.")

## 5 · The point of the whole tutorial: baselines & selectivity

High accuracy means nothing on its own. A probe on *random* features can look good; an *expressive*
probe can score high by memorising rather than by reading the representation. We defend against this
with three baselines and one derived number.

- **Majority class** — always predict the commonest label. The floor.
- **Random representation** — the *same* probe on activations from an **untrained** model. Isolates
  what training the encoder actually added.
- **Control task** (Hewitt & Liang, 2019) — the same probe on **randomised labels**. This measures the
  probe's own *memorisation capacity*. (The original control task randomises labels per word *type*
  for token tasks; for sentence classification we randomise example labels, which plays the same role.)

**Selectivity = accuracy(real labels) − accuracy(control).** High selectivity = the representation is
doing work the probe alone cannot.

### ✏️ Exercise 4a — the random-representation baseline
Build an **untrained** GPT-2 of the same size (random weights) and extract activations from *it*, then
train/evaluate the same linear probe. If trained ≈ random, training taught the model nothing useful
*for this concept*.

In [ ]:
def build_random_model():
    cfg = AutoConfig.from_pretrained(MODEL_NAME)
    rnd = AutoModel.from_config(cfg).to(device).eval()   # random init, same architecture
    for p in rnd.parameters(): p.requires_grad_(False)
    return rnd

_trained_model = model
def activations_from(m, texts, layer):
    global model
    model = m
    try:
        return get_activations(texts, layer)
    finally:
        model = _trained_model

rnd_model = build_random_model()
Xtr_rand = activations_from(rnd_model, X_train, LAYER)
Xte_rand = activations_from(rnd_model, X_test,  LAYER)
rand_probe = train_linear_probe(Xtr_rand, y_train)
rand_acc = accuracy(rand_probe, Xte_rand, y_test)


In [ ]:
print(f"trained-model probe : {acc:.3f}")
print(f"random-model probe  : {rand_acc:.3f}")
print(f"majority baseline   : {max(np.mean(y_test), 1-np.mean(y_test)):.3f}")
assert 0.0 <= rand_acc <= 1.0
print("✓ Exercise 4a — how much did *training* the encoder add over random features?")

### ✏️ Exercise 4b — the control task and selectivity
Randomly permute the training labels, train the *same* probe, and evaluate on the *real* test labels —
its accuracy is roughly the probe's ability to fit noise (chance-ish for a linear probe). Then compute
selectivity.

In [ ]:
def selectivity(Xtr, ytr, Xte, yte, C=1.0):
    real = accuracy(train_linear_probe(Xtr, ytr, C), Xte, yte)
    y_shuf = np.random.permutation(ytr)
    ctrl = accuracy(train_linear_probe(Xtr, y_shuf, C), Xte, yte)
    return real, ctrl, real - ctrl

real, ctrl, sel = selectivity(Xtr, y_train, Xte, y_test)


In [ ]:
print(f"real accuracy    : {real:.3f}")
print(f"control accuracy : {ctrl:.3f}")
print(f"SELECTIVITY      : {sel:.3f}   (real - control)")
assert sel > 0.1, "a linear sentiment probe should have clearly positive selectivity"
print("✓ Exercise 4b")

## 6 · The trap: a stronger probe that tells you *less*

Swap the linear probe for a small **MLP**. It will usually score **higher accuracy** — but watch the
**selectivity**. If the MLP's control accuracy also jumps, the extra accuracy came from the *probe's*
capacity to memorise, not from more signal in the representation. This is exactly why "our MLP got
97%!" is not, by itself, evidence about the model.

### ✏️ Exercise 5 — compare linear vs MLP on *both* accuracy and selectivity

In [ ]:
from sklearn.neural_network import MLPClassifier

def train_mlp_probe(X, y, hidden=256):
    probe = make_pipeline(StandardScaler(),
                          MLPClassifier(hidden_layer_sizes=(hidden,), max_iter=300,
                                        random_state=SEED))
    probe.fit(X, y)
    return probe

def selectivity_mlp(Xtr, ytr, Xte, yte, hidden=256):
    real = accuracy(train_mlp_probe(Xtr, ytr, hidden), Xte, yte)
    ctrl = accuracy(train_mlp_probe(Xtr, np.random.permutation(ytr), hidden), Xte, yte)
    return real, ctrl, real - ctrl


In [ ]:
lin = selectivity(Xtr, y_train, Xte, y_test)
mlp = selectivity_mlp(Xtr, y_train, Xte, y_test)
print(f"{'probe':10} {'accuracy':>10} {'control':>10} {'selectivity':>12}")
print(f"{'linear':10} {lin[0]:>10.3f} {lin[1]:>10.3f} {lin[2]:>12.3f}")
print(f"{'MLP':10} {mlp[0]:>10.3f} {mlp[1]:>10.3f} {mlp[2]:>12.3f}")
print("\n→ Which probe is more ACCURATE? Which is more SELECTIVE? Which would you report, and why?")
print("✓ Exercise 5")

## 7 · Accessible ≠ used — the deepest limitation

Everything so far shows a concept is **accessible** *from* $h$. It does **not** show the model **uses**
that direction. These are different claims. We probe both sides of it.

**(a) The concept lives in a direction.** Take the probe's weight vector $w$, make it unit length, and
**project it out** of the activations: $h' = h - (\hat w \cdot h)\,\hat w$. If the probe can no longer
read sentiment from $h'$, the information really was concentrated along that direction (this is the
idea behind INLP / amnesic probing).

### ✏️ Exercise 6 — ablate the concept direction

In [ ]:
def probe_direction(probe):
    # Unit-norm weight vector of the LogisticRegression inside the pipeline (standardised space).
    lr = probe.named_steps["logisticregression"]
    w = lr.coef_[0]
    return w / (np.linalg.norm(w) + 1e-8)

def ablate_direction(X, probe):
    scaler = probe.named_steps["standardscaler"]
    Xs = scaler.transform(X)
    w = probe_direction(probe)
    proj = (Xs @ w)[:, None] * w[None, :]     # component of each row along w
    return Xs - proj                          # remove it


In [ ]:
# a fresh probe operating directly on standardised space, so we can feed ablated features back in
from sklearn.linear_model import LogisticRegression as _LR
lr_only = _LR(max_iter=2000).fit(probe.named_steps["standardscaler"].transform(Xtr), y_train)
Xte_std   = probe.named_steps["standardscaler"].transform(Xte)
Xte_abl   = ablate_direction(Xte, probe)
acc_before = float((lr_only.predict(Xte_std) == y_test).mean())
acc_after  = float((lr_only.predict(Xte_abl) == y_test).mean())
print(f"probe accuracy before ablation: {acc_before:.3f}")
print(f"probe accuracy after  ablation: {acc_after:.3f}")
assert acc_after < acc_before - 0.1, "removing the direction should hurt decodability"
print("✓ Exercise 6 — the concept was concentrated along one direction.")

**(b) Is that direction *causally used*?** Removing it from the probe's input only shows
*accessibility*. The honest causal test is to intervene on the **running model** and see the **output**
move. Below (provided) we do exactly that with **steering**: add $\alpha\,\hat w$ to the residual
stream *during generation* and watch continuations shift. When the output changes, the direction is
being used from that layer.

> This is a light, tangible version of the causal tests in the lecture (activation patching /
> interchange interventions). The full versions swap activations between clean and corrupted runs and
> measure the change in the answer token — that, not probe accuracy, is what licenses a causal claim.

In [ ]:
# STEERING (provided): recover a sentiment direction in the model's *raw* activation space,
# then add it back during generation via a forward hook on a transformer block.
gen = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device).eval()
for p in gen.parameters(): p.requires_grad_(False)

# direction in RAW space = difference of class means at LAYER (simple & robust for steering)
mu_pos = Xtr[y_train == 1].mean(0); mu_neg = Xtr[y_train == 0].mean(0)
steer_vec = torch.tensor((mu_pos - mu_neg), dtype=torch.float32, device=device)
steer_vec = steer_vec / steer_vec.norm()

def make_hook(alpha):
    def hook(module, inp, out):
        h = out[0] if isinstance(out, tuple) else out
        h = h + alpha * steer_vec
        return (h,) + out[1:] if isinstance(out, tuple) else h
    return hook

@torch.no_grad()
def generate(prompt, alpha=0.0, layer=LAYER, n=40):
    handle = gen.transformer.h[layer-1].register_forward_hook(make_hook(alpha))
    try:
        ids = tokenizer(prompt, return_tensors="pt").to(device)
        out = gen.generate(**ids, max_new_tokens=n, do_sample=False,
                           pad_token_id=tokenizer.eos_token_id)
        return tokenizer.decode(out[0], skip_special_tokens=True)
    finally:
        handle.remove()

prompt = "The movie was"
for a in (-14.0, 0.0, +14.0):
    print(f"\n[alpha={a:+.0f}] {generate(prompt, a)!r}")
print("\n→ Does the sentiment of the continuation move with alpha? (Small model → effect is rough.)")

## 8 · The real thing: probing for truthfulness / deception

Safety probes (e.g. Apollo Research's *Detecting Strategic Deception Using Linear Probes*, 2025;
Anthropic's *sleeper-agent* probes) are **exactly this pipeline** pointed at honesty: train a linear
probe on activations from **honest vs deceptive / true vs false** text, and monitor it at deployment.

Here we use a small set of **true vs false factual statements** and ask: does GPT-2 *linearly encode*
whether a statement is true? Then you apply the discipline you just learned — baseline **and**
selectivity — because a "lie detector" that isn't selective is worse than useless.

### ✏️ Exercise 7 — a truthfulness probe, evaluated honestly

In [ ]:
TRUE = ["The capital of France is Paris.", "Water is made of hydrogen and oxygen.",
        "The Earth orbits the Sun.", "Two plus two equals four.",
        "The Pacific is the largest ocean.", "Humans have two lungs.",
        "Ice is frozen water.", "A triangle has three sides.",
        "The Sun rises in the east.", "Mount Everest is the tallest mountain.",
        "Spiders have eight legs.", "The heart pumps blood.",
        "December has thirty-one days.", "Gold is a metal.",
        "A week has seven days.", "The moon orbits the Earth."]
FALSE = ["The capital of France is Berlin.", "Water is made of gold and helium.",
         "The Sun orbits the Earth.", "Two plus two equals five.",
         "The Atlantic is the largest ocean.", "Humans have three lungs.",
         "Ice is boiling water.", "A triangle has four sides.",
         "The Sun rises in the west.", "Mount Everest is the shortest mountain.",
         "Spiders have six legs.", "The heart pumps air.",
         "December has forty days.", "Gold is a gas.",
         "A week has ten days.", "The moon orbits Mars."]
texts_tf = TRUE + FALSE
y_tf = np.array([1]*len(TRUE) + [0]*len(FALSE))
# a tiny split
idx = np.random.permutation(len(texts_tf)); cut = int(0.7*len(idx))
tr_i, te_i = idx[:cut], idx[cut:]
Xtf = get_activations(texts_tf, LAYER)
print(f"{len(tr_i)} train / {len(te_i)} test true/false statements")

In [ ]:
tf_probe = train_linear_probe(Xtf[tr_i], y_tf[tr_i])
tf_acc = accuracy(tf_probe, Xtf[te_i], y_tf[te_i])
tf_base = max(np.mean(y_tf[te_i]), 1 - np.mean(y_tf[te_i]))
_, _, tf_sel = selectivity(Xtf[tr_i], y_tf[tr_i], Xtf[te_i], y_tf[te_i])
print(f"truthfulness probe accuracy : {tf_acc:.3f}")
print(f"majority baseline           : {tf_base:.3f}")
print(f"selectivity                 : {tf_sel:.3f}")


**Interpret it like a scientist.** With only ~30 statements on a 124M model these numbers are
noisy and modest — that is the honest result, and saying so is the skill. Real deception probes reach
AUROC 0.96–0.999 (Apollo) on large instruct models with hundreds of contrast pairs. But the same
caveats bite even there:

- Is the probe detecting *deception*, or merely *deception-flavoured text*? (Apollo find some of the
  latter — this is the **selectivity / readout** problem you met in §5–6.)
- Does it **generalise** to a setting it wasn't trained on? (Usually degrades — the *where probes
  break* slide.)
- Does firing on activations tell you more than reading the transcript would? (If a black-box LLM
  judge does as well, the probe hasn't surfaced hidden cognition — *accessible ≠ used*, §7.)

## 9 · Capstone & the checklist

You now have every tool from the lecture. Design a small study of your own and report it honestly.

**Pick a concept** (topic, formality, tense, presence of a named entity, toxicity, …), build a
labelled set, and run the full pipeline:

1. **Freeze** the encoder. (Always.)
2. **Linear probe first**, every layer — plot the profile.
3. **All three baselines** — majority, random-representation, control — and **selectivity**.
4. Only then try an **MLP**, and justify it against selectivity.
5. If you want a *causal* claim, **intervene** (ablate / steer / patch) — probe accuracy is not enough.
6. **State the limits.**

### The report template (from the lecture)
> *"We trained a linear probe (logistic regression) on mean-pooled activations from all layers of GPT-2
> to predict [concept]. The probe reached X% accuracy (majority Y%, random-representation Z%, control
> W%, selectivity X−W%), peaking at layer L. This indicates [concept] is linearly decodable at layer L
> with selectivity S%, suggesting it is encoded beyond what the probe could memorise. We did / did not
> establish that the model uses this direction (intervention result)."*

**Language matters:** "linearly decodable", not "the model knows". "suggests", not "proves".

---
### Seven things to remember
1. Freeze the encoder — always.
2. Linear probe first; justify any step up in complexity.
3. Linear ≠ nonlinear: a *direction* is a stronger claim than *decodable*.
4. All three baselines; never raw accuracy alone.
5. Probe every layer — the profile is the finding.
6. Accuracy ≠ causation: probing shows access, intervention shows use.
7. State your limits.

*Further reading:* the lecture slides (`OXML_Summer_School_2026_Lecture.pdf`), Apollo Research —
*Detecting Strategic Deception Using Linear Probes* (2025), Hewitt & Liang — *Designing and
Interpreting Probes with Control Tasks* (2019), and EleutherAI's `mdl` repo for the
minimum-description-length version of §5.